In [2]:
from pathlib import Path
import sys

ROOT_DIR = Path().resolve().parents[1]
sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from config.paths import CLEANED_MARKET_FILE

df = pd.read_csv(CLEANED_MARKET_FILE)

In [3]:
df

,date,symbol,open,high,low,close,volume
0,2005-02-22 10:00:00,EGX:ABUK,3.272724,3.287270,3.272724,3.273815,2.308627e+05
1,2005-02-23 10:00:00,EGX:ABUK,3.272724,3.309088,3.272724,3.283997,2.640002e+05
2,2005-02-24 10:00:00,EGX:ABUK,3.272724,3.341815,3.272724,3.306542,3.169928e+05
3,2005-02-27 10:00:00,EGX:ABUK,3.345452,3.471633,3.345452,3.407633,1.337354e+06
4,2005-02-28 10:00:00,EGX:ABUK,3.452724,3.454542,3.381815,3.404361,6.105005e+05
...,...,...,...,...,...,...,...
119026,2026-02-15 10:00:00,EGX:TMGH,88.870000,92.890000,88.510000,91.600000,3.417900e+06
119027,2026-02-16 10:00:00,EGX:TMGH,91.600000,95.350000,91.600000,92.000000,6.168783e+06
119028,2026-02-17 10:00:00,EGX:TMGH,92.000000,97.000000,92.000000,95.700000,4.753944e+06
119029,2026-02-18 10:00:00,EGX:TMGH,95.700000,96.800000,95.540000,96.370000,1.605437e+06


In [4]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

In [5]:
recent_3_years = [2023, 2024, 2025]
filtered_df = df[df['date'].dt.year.isin(recent_3_years)]

filtered_df

,date,symbol,open,high,low,close,volume
53482,2023-01-02 00:00:00,EGX:ESRS,23.250000,25.520000,23.400000,25.450001,2974334.0
32518,2023-01-02 00:00:00,EGX:COMI,39.652780,41.000669,39.432912,40.866837,2708615.0
51010,2023-01-02 00:00:00,EGX:EKHOA,31.437703,32.554778,31.527468,32.514881,519106.0
76712,2023-01-02 10:00:00,EGX:ISPH,2.160000,2.250000,2.160000,2.210000,10451359.0
4241,2023-01-02 10:00:00,EGX:ABUK,38.790001,40.889999,38.750000,40.820000,1852361.0
...,...,...,...,...,...,...,...
104844,2025-12-31 10:00:00,EGX:PHDC,8.510000,8.670000,8.500000,8.610000,8464156.0
70448,2025-12-31 10:00:00,EGX:HELI,3.620000,3.640000,3.570000,3.600000,15934532.0
30768,2025-12-31 10:00:00,EGX:CIRA,17.520000,17.600000,17.250000,17.370000,69179.0
114597,2025-12-31 10:00:00,EGX:SWDY,77.010000,78.620000,77.010000,78.250000,451471.0


In [6]:
monthly_df = df.set_index('date').groupby('symbol')['close'].resample('ME').last().reset_index()
monthly_df['monthly_return'] = monthly_df.groupby('symbol')['close'].pct_change()
monthly_df.dropna(inplace=True)

monthly_df

C:\Users\nasho\AppData\Local\Temp\ipykernel_21980\580453813.py:2: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  monthly_df['monthly_return'] = monthly_df.groupby('symbol')['close'].pct_change()


,symbol,date,close,monthly_return
1,EGX:ABUK,2005-03-31,3.442906,0.011322
2,EGX:ABUK,2005-04-30,3.564360,0.035277
3,EGX:ABUK,2005-05-31,3.364361,-0.056111
4,EGX:ABUK,2005-06-30,3.547997,0.054583
5,EGX:ABUK,2005-07-31,3.535270,-0.003587
...,...,...,...,...
6552,EGX:TMGH,2025-10-31,57.510000,0.011076
6553,EGX:TMGH,2025-11-30,76.440000,0.329160
6554,EGX:TMGH,2025-12-31,80.000000,0.046572
6555,EGX:TMGH,2026-01-31,86.490000,0.081125


In [7]:
stocks = monthly_df.groupby('symbol')['monthly_return'].agg([
    ('avg_annual_return', lambda x: x.mean() * 12),
    ('annualized_sd', lambda x: x.std() * np.sqrt(12))
]).reset_index()

stocks

,symbol,avg_annual_return,annualized_sd
0,EGX:ABUK,0.198161,0.340231
1,EGX:ADIB,0.310463,0.541222
2,EGX:AMOC,0.079035,0.371944
3,EGX:BTFH,0.462965,0.859340
4,EGX:CCAP,0.071999,0.518676
5,EGX:CIEB,1.026375,3.698223
6,EGX:CIRA,0.213097,0.561032
7,EGX:COMI,0.298235,0.320661
8,EGX:DOMT,0.067304,0.581641
9,EGX:EAST,0.249826,0.358519


In [8]:
stocks['risk_level'] = pd.qcut(
    stocks['annualized_sd'], 
    q=3, 
    labels=['Conservative', 'Moderate', 'Aggressive']
)

In [9]:
stocks = stocks.sort_values(by=['risk_level', 'avg_annual_return'], ascending=[True, False])

print(stocks)

       symbol  avg_annual_return  annualized_sd    risk_level
7    EGX:COMI           0.298235       0.320661  Conservative
28   EGX:SWDY           0.269158       0.418153  Conservative
9    EGX:EAST           0.249826       0.358519  Conservative
20   EGX:JUFO           0.237487       0.413050  Conservative
0    EGX:ABUK           0.198161       0.340231  Conservative
15   EGX:EXPA           0.192851       0.403173  Conservative
25   EGX:ORWE           0.142886       0.380080  Conservative
14   EGX:ETEL           0.125377       0.309281  Conservative
12  EGX:EKHOA           0.111389       0.304016  Conservative
2    EGX:AMOC           0.079035       0.371944  Conservative
16   EGX:FWRY           0.427635       0.443187      Moderate
22   EGX:MFPC           0.394408       0.478790      Moderate
23   EGX:MTIE           0.283537       0.521705      Moderate
10   EGX:EFIC           0.259015       0.487675      Moderate
21   EGX:MASR           0.253175       0.530209      Moderate
29   EGX

In [ ]:
stocks = stocks.drop(columns=['avg_annual_return', 'annualized_sd'])
stocks.to_csv(ROOT_DIR / 'data' /'database_data' / 'stocks.csv', index=False)